## Train File

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys

sys.path.append('../src')

import util as ut

In [2]:
df_train = pd.read_csv("../data/processed/01.2/train_data.csv")
df_test = pd.read_csv("../data/processed/01.2/test_data.csv")

In [3]:
df_train["Start_Time"] = pd.to_datetime(df_train["Start_Time"], errors="coerce")

In [4]:
missing_data = df_train.isnull().sum()[df_train.isnull().sum() > 0].sort_values(ascending=False)

print("--- Number of Null ---")
print(missing_data)

--- Number of Null ---
Precipitation(in)    1645148
Wind_Chill(F)        1482771
Wind_Speed(mph)       344794
Wind_Direction         33784
Visibility(mi)         32958
Weather_Condition      30777
Humidity(%)            29692
Temperature(F)         22357
dtype: int64


In [5]:
missing_percent = (df_train.isnull().sum() / len(df_train)) * 100
missing_df = missing_percent[missing_percent > 0].sort_values(ascending=False)

print("--- Percent of Null ---")
print(missing_df)

--- Percent of Null ---
Precipitation(in)    30.080825
Wind_Chill(F)        27.111831
Wind_Speed(mph)       6.304410
Wind_Direction        0.617726
Visibility(mi)        0.602623
Weather_Condition     0.562744
Humidity(%)           0.542905
Temperature(F)        0.408788
dtype: float64


#### Impute Weather Condition

In [6]:
print(df_train["Weather_Condition"].unique())
print(df_train["Weather_Condition"].nunique())

['Overcast' 'Fair' 'Clear' 'Mostly Cloudy' 'Partly Cloudy' 'Thunder'
 'Thunderstorm' 'Light Rain' 'Cloudy' 'Light Rain with Thunder' 'Fog'
 'Light Snow' 'Scattered Clouds' 'Light Drizzle' 'N/A Precipitation'
 'Haze' nan 'Thunder in the Vicinity' 'Snow' 'Fair / Windy' 'Drizzle'
 'Rain' 'Heavy Rain' 'Light Thunderstorms and Rain' 'Light Rain / Windy'
 'Mostly Cloudy / Windy' 'Cloudy / Windy' 'Heavy Snow / Windy'
 'Heavy Rain / Windy' 'Wintry Mix' 'Patches of Fog' 'Mist' 'Shallow Fog'
 'Heavy Snow' 'T-Storm' 'Smoke' 'Rain / Windy' 'Blowing Snow'
 'Snow / Windy' 'Light Snow / Windy' 'Fog / Windy' 'Partly Cloudy / Windy'
 'Light Freezing Rain' 'Heavy T-Storm' 'Heavy Thunderstorms and Rain'
 'Thunderstorms and Rain' 'Blowing Dust' 'Wintry Mix / Windy'
 'Light Freezing Fog' 'Showers in the Vicinity' 'Haze / Windy'
 'Light Freezing Drizzle' 'Thunder / Windy' 'Blowing Snow / Windy'
 'Light Rain Showers' 'Light Ice Pellets' 'Light Drizzle / Windy'
 'Light Snow with Thunder' 'Light Snow and Sleet

In [7]:
df_train['Weather_Condition'] = df_train['Weather_Condition'].str.lower()

In [8]:
df_train['Start_Date'] = df_train['Start_Time'].dt.date
df_train['Hour'] = df_train['Start_Time'].dt.hour
df_train['Month'] = df_train['Start_Time'].dt.month

In [10]:
weather_first = df_train.groupby(['City', 'Start_Date', 'Hour', 'Street'])['Weather_Condition'].transform('first')
df_train['Weather_Condition'] = df_train['Weather_Condition'].fillna(weather_first)

In [11]:
df_train["Weather_Condition"].isnull().sum()

np.int64(30517)

#### Analyze Weather Condition Keywords
วิเคราะห์คำที่ปรากฏบ่อยที่สุดใน Weather_Condition เพื่อหา Keyword ในการจัดกลุ่ม

In [9]:
from collections import Counter
import re

# 1. นับจำนวนประโยคเต็มที่เจอบ่อยที่สุด
print("--- Top 20 Full Weather Conditions ---")
print(df_train['Weather_Condition'].value_counts().head(20))

# 2. แยกคำ (Tokenize) และนับความถี่ของแต่ละคำ
all_text = " ".join(df_train['Weather_Condition'].dropna().astype(str).str.lower())
words = re.findall(r'\w+', all_text)
word_counts = Counter(words)

print("\n--- Top 50 Most Frequent Words in Weather_Condition ---")
for word, count in word_counts.most_common(50):
    print(f"{word:15}: {count:,}")

# 3. เช็กว่ามีคำไหนบ้างที่ยังไม่ถูกจับคู่ในกลุ่มที่มีอยู่ (Optional)
existing_keywords = 'thunder|storm|tornado|gale|snow|sleet|ice|freez|fog|haze|smoke|dust|rain|drizzle|shower|wind|breezy|blustery|gusty|cloud|overcast|clear|fair'
unmatched_conditions = df_train[~df_train['Weather_Condition'].str.contains(existing_keywords, na=False, case=False)]['Weather_Condition'].unique()

print(f"\n--- Samples of Unmatched Conditions ({len(unmatched_conditions)} unique values) ---")
print(unmatched_conditions[:20])

--- Top 20 Full Weather Conditions ---
Weather_Condition
fair                       1752683
mostly cloudy               736784
clear                       645566
cloudy                      568464
partly cloudy               505899
overcast                    305986
light rain                  256422
scattered clouds            163722
light snow                   89369
fog                          70493
rain                         61039
haze                         56333
fair / windy                 24045
heavy rain                   23434
light drizzle                16737
thunder in the vicinity      11721
cloudy / windy               11634
t-storm                      11390
mostly cloudy / windy        11276
snow                         10568
Name: count, dtype: int64

--- Top 50 Most Frequent Words in Weather_Condition ---
cloudy         : 1,841,025
fair           : 1,776,728
mostly         : 748,060
clear          : 645,566
partly         : 512,867
light          : 390,979
rain  

In [ ]:
import numpy as np

conditions = [
    # 1. ระดับวิกฤต
    df_train['Weather_Condition'].str.contains('thunder|storm|tornado|gale', na=False),
    
    # 2. ถนนลื่นอันตรายมาก
    df_train['Weather_Condition'].str.contains('snow|sleet|ice|freez', na=False),
    
    # 3. ทัศนวิสัยแย่มาก 
    df_train['Weather_Condition'].str.contains('fog|haze|smoke|dust', na=False),
    
    # 4. ถนนลื่นปกติ
    df_train['Weather_Condition'].str.contains('rain|drizzle|shower', na=False),
    
    # 5. ลมแรง
    df_train['Weather_Condition'].str.contains('wind|breezy|blustery|gusty', na=False),
    
    # 6. ท้องฟ้ามืดครึ้ม
    df_train['Weather_Condition'].str.contains('cloud|overcast', na=False),
    
    # 7. ท้องฟ้าแจ่มใส
    df_train['Weather_Condition'].str.contains('clear|fair', na=False)
]

choices = [
    'Severe',   # 1
    'Snow_Ice', # 2
    'Fog',      # 3
    'Rain',     # 4
    'Windy',    # 5
    'Cloudy',   # 6
    'Clear'     # 7
]

df_train['Weather_Group'] = np.select(conditions, choices, default='Other')

df_train = df_train.drop(columns=['Weather_Condition'])

In [ ]:
print(df_train["Weather_Group"].value_counts())

Weather_Group
Clear       2399415
Cloudy      2282443
Rain         371351
Fog          144237
Snow_Ice     115511
Severe        61735
Windy         54216
Other         40184
Name: count, dtype: int64


In [ ]:
mask_other = df_train["Weather_Group"].isin(["Other", np.nan])

conditions_from_numbers = [
    (((df_train['Wind_Speed(mph)'].notnull()) & (df_train['Wind_Speed(mph)'] > 40)) | 
    ((df_train['Precipitation(in)'].notnull()) & (df_train['Precipitation(in)'] > 1.0))),


    (df_train['Precipitation(in)'].notnull()) & (df_train['Temperature(F)'].notnull()) & 
    (df_train['Precipitation(in)'] > 0) & (df_train['Temperature(F)'] < 32),
    
    (df_train['Precipitation(in)'].notnull()) & (df_train['Temperature(F)'].notnull()) & 
    (df_train['Precipitation(in)'] > 0) & (df_train['Temperature(F)'] >= 32),
    
    (df_train['Visibility(mi)'].notnull()) & 
    (df_train['Visibility(mi)'] < 2.0),
    
    (df_train['Wind_Speed(mph)'].notnull()) & 
    (df_train['Wind_Speed(mph)'] > 20),
    
    (df_train['Humidity(%)'].notnull()) & 
    (df_train['Humidity(%)'] > 80),
    
    (df_train['Temperature(F)'].notnull())
]

choices_for_numbers = [
    "Severe",  # 1
    "Snow_Ice",  # 2
    "Rain",  # 3
    "Fog",  # 4
    "Windy",  # 5
    "Cloudy",  # 6
    "Clear",  # 7
]

df_train.loc[mask_other, "Weather_Group"] = np.select(
    [cond[mask_other] for cond in conditions_from_numbers],
    choices_for_numbers,
    default="Other",
)

print(df_train["Weather_Group"].value_counts())

Weather_Group
Clear       2416961
Cloudy      2293142
Rain         376980
Fog          146758
Snow_Ice     115513
Severe        61735
Windy         54216
Other          3787
Name: count, dtype: int64


In [ ]:
# Hierarchical Imputation
from pandas.api.types import is_numeric_dtype, is_string_dtype, is_object_dtype

hierarchies = [
    ['City', 'Start_Date', 'Hour'],  
    ['City', 'Start_Date'],        
    ['City', 'Month'],           
    ['Weather_Group']            
]

num_weather_cols = [
    'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 
    'Pressure(in)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)', "Wind_Direction"
]

for col in num_weather_cols:
    
    for group_cols in hierarchies:

        if df_train[col].isnull().sum() == 0:
            break
        
        if is_numeric_dtype(df_train[col]):
            df_train[col] = df_train[col].fillna(
                df_train.groupby(group_cols)[col].transform('median')
            )
            
        elif is_string_dtype(df_train[col]) or is_object_dtype(df_train[col]):
            df_train[col] = df_train[col].fillna(
                df_train.groupby(group_cols)[col].transform('first')
            )

เริ่มกระบวนการ Hierarchical Imputation...


In [ ]:
missing_percent = (df_train.isnull().sum() / len(df_train)) * 100
missing_df = missing_percent[missing_percent > 0].sort_values(ascending=False)

print("--- Percent of Null ---")
print(missing_df)

--- Percent of Null ---
Series([], dtype: float64)


In [ ]:
save_dir = "../data/processed/01.3"

df_train.to_csv("../data/processed/01.3/accidents_advance_clean.csv", index=False)

print(f"Number of df_adv_clean rows: {df_train.shape[0]:,} rows")

Number of df_adv_clean rows: 5,469,092 rows
